# Submission 03 - Hist Gradient Boosting

This submission uses the Hist Gradient Boosting configuration from `10_hist_gradient_boosting.ipynb`, which achieved 0.7933 validation accuracy.

The model is retrained on the full training dataset before generating the Kaggle submission.


In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier


In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

y = train['Survived']
test_ids = test['PassengerId'].copy()


In [3]:
def feature_engineering(df):
    df = df.copy()

    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['Mother'] = ((df['Sex'] == 'female') & (df['Parch'] > 0) & (df['Age'] > 18) & (df['Parch'] < 5)).astype(int)
    df['Child'] = (df['Age'] < 14).astype(int)
    df['AgeMissing'] = df['Age'].isna().astype(int)
    df['FareMissing'] = df['Fare'].isna().astype(int)
    df['EmbarkedMissing'] = df['Embarked'].isna().astype(int)
    df['HasCabin'] = df['Cabin'].notna().astype(int)
    df['CabinDeck'] = df['Cabin'].fillna('U').str[0]

    df['Title'] = df['Name'].str.extract(r',\s*([^.]+)\.', expand=False).str.strip()
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    common_titles = ['Mr', 'Miss', 'Mrs', 'Master']
    df.loc[~df['Title'].isin(common_titles), 'Title'] = 'Rare'

    df['Surname'] = df['Name'].str.split(',').str[0].str.strip()
    df['TicketPrefix'] = df['Ticket'].str.replace(r'\d', '', regex=True).str.replace(r'[./]', '', regex=True).str.replace(' ', '', regex=True).replace('', 'NONE')
    df['TicketGroupSize'] = df.groupby('Ticket')['Ticket'].transform('count')
    df['SurnameGroupSize'] = df.groupby('Surname')['Surname'].transform('count')
    df['FarePerPerson'] = df['Fare'] / df['TicketGroupSize'].replace(0, 1)
    df['SexPclass'] = df['Sex'].astype(str) + '_' + df['Pclass'].astype(str)
    df['FamilySizeBand'] = pd.cut(df['FamilySize'], bins=[0, 1, 4, 7, 100], labels=['Alone', 'Small', 'Medium', 'Large'])
    df['AgeBand'] = pd.cut(df['Age'], bins=[-1, 5, 12, 18, 30, 45, 60, 100], labels=['Baby', 'Child', 'Teen', 'YoungAdult', 'Adult', 'MiddleAge', 'Senior'])
    df['FareBand'] = pd.qcut(df['Fare'].rank(method='first'), 5, labels=['VeryLow', 'Low', 'Medium', 'High', 'VeryHigh'])
    df['FamilySex'] = df['Sex'].astype(str) + '_' + df['FamilySizeBand'].astype(str)
    df['PclassTitle'] = df['Pclass'].astype(str) + '_' + df['Title'].astype(str)
    df['PclassAgeBand'] = df['Pclass'].astype(str) + '_' + df['AgeBand'].astype(str)
    df['FamilyTicket'] = df['FamilySize'].astype(str) + '_' + df['TicketPrefix'].astype(str)
    df['FarePerPersonMissing'] = df['FarePerPerson'].isna().astype(int)
    df['NameLength'] = df['Name'].str.len()
    df['NameWords'] = df['Name'].str.split().str.len()
    df['TicketLength'] = df['Ticket'].str.len()
    df['CabinCount'] = df['Cabin'].fillna('').str.split().str.len()
    df['DeckKnown'] = (df['CabinDeck'] != 'U').astype(int)
    df['LargeFamily'] = (df['FamilySize'] >= 5).astype(int)
    df['SmallFamily'] = df['FamilySize'].between(2, 4).astype(int)
    df['FemaleChild'] = ((df['Sex'] == 'female') | (df['Age'] < 14)).astype(int)
    df['FarePerAge'] = df['Fare'] / (df['Age'].fillna(df['Age'].median()) + 1)
    df['ClassFare'] = df['Fare'] / df['Pclass']
    df['SiblingChildRatio'] = df['SibSp'] / (df['Parch'] + 1)
    df['FamilyFare'] = df['Fare'] * df['FamilySize']
    df['SexTitle'] = df['Sex'].astype(str) + '_' + df['Title'].astype(str)

    return df


In [4]:
train_fe = feature_engineering(train.drop(columns=['Survived']))
test_fe = feature_engineering(test.copy())

drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Surname']

X = train_fe.drop(columns=drop_cols, errors='ignore')
X_test = test_fe.drop(columns=drop_cols, errors='ignore')

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()


In [5]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])


In [6]:
model = HistGradientBoostingClassifier(
    max_iter=400,
    learning_rate=0.04,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])


In [7]:
# Train on the full training dataset
pipeline.fit(X, y)

test_pred = pipeline.predict(X_test).astype(int)


In [8]:
submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': test_pred
})

assert submission.shape == (418, 2)
assert list(submission.columns) == ['PassengerId', 'Survived']
assert submission['PassengerId'].is_unique
assert submission['Survived'].isin([0, 1]).all()

output_path = '../submissions/submission_03.csv'
submission.to_csv(output_path, index=False)

print(f'Saved: {output_path}')
print(f'Rows: {len(submission)}')
print(f'Survived predictions: {submission["Survived"].sum()}')
print(f'Died predictions: {(submission["Survived"] == 0).sum()}')
display(submission.head(10))


Saved: ../submissions/submission_03.csv
Rows: 418
Survived predictions: 164
Died predictions: 254


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,0
7,899,0
8,900,1
9,901,0
